# MyAI 100M Train — Leg 3 (GPU)
Resumes from leg 2 final.pt. Stops at step 12750 of the shared 15000-step cosine target. ~2250 steps (~1.1h GPU).

In [ ]:
!git clone https://github.com/aloksingh9667/llm.git myai 2>&1 | tail -1
%cd /kaggle/working/myai
!pip install -q -r requirements/train.txt 2>&1 | tail -1

In [ ]:
!find /kaggle/input -name 'final.pt' 2>/dev/null
!test -f /kaggle/input/myai-fwab-32k/manifest.json && echo DATA_OK
!test -f /kaggle/input/myai-fwab-32k/tokenizer-32k.json && echo TOKENIZER_OK
!python -c "import json; m=json.load(open('/kaggle/input/myai-fwab-32k/manifest.json')); print('halves', m['halves'], 'train_rows', m['train_sequences'], 'val_rows', m['val_sequences'])"

In [ ]:
%%bash
cd /kaggle/working/myai
CKPT=$(find /kaggle/input -name 'final.pt' 2>/dev/null | head -1)
test -n "${CKPT}" || { echo NO_CKPT; exit 1; }
echo "resume-from: ${CKPT}"
python scripts/train.py --model-config configs/myai-100m.yaml \
    --data-dir /kaggle/input/myai-fwab-32k \
    --max-steps 15000 --stop-step 12750 \
    --batch-size 4 --accum 8 --amp --no-shuffle \
    --eval-every 500 --ckpt-every 2500 \
    --resume "${CKPT}" \
    --out-dir /kaggle/working/ckpts-leg3 \
    --report /kaggle/working/train-leg3.json

In [ ]:
!test -f /kaggle/working/ckpts-leg3/final.pt && echo LEG3_OK
!python -c "import json; r=json.load(open('/kaggle/working/train-leg3.json')); [print(k, r.get(k)) for k in ('steps','first_loss','final_train_loss','val_loss','tokens_seen')]"